In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="transformers",
    notebook_name="01_attention_mechanisms_experiments.ipynb"
)

# Experiments: Attention Mechanisms

This notebook produces **runnable evidence** for claims you'll make in interviews about attention. Each experiment tests a specific claim and gives you real numbers to cite.

Before running this notebook, make sure you've read [attention-mechanisms.md](./attention-mechanisms.md) and worked through [01_attention_mechanisms.ipynb](./01_attention_mechanisms.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)

## Setup: Reuse attention from the concept notebook

We need the same attention function we built in the concept notebook.

In [ ]:
def softmax(x):
    """Numerically stable softmax along last axis."""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute attention: softmax(QK^T / sqrt(d_k)) * V"""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask  # mask has -inf where attention is blocked
    weights = softmax(scores)
    output = weights @ V
    return output, weights


print("Attention function ready.")

---

## Experiment 1: Complexity Benchmark — O(n^2) Scaling

**Claim to test:** Attention is O(n^2 * d_k) in compute and O(n^2) in memory.

**Why it matters in an interview:** When asked about transformer limitations, you need to cite the quadratic scaling with real numbers — not just say "O(n squared)".

In [ ]:
# Measure runtime of attention at increasing sequence lengths
d_k = 64
seq_lengths = [32, 64, 128, 256, 512, 1024, 2048]
times = []

for n in seq_lengths:
    Q = np.random.randn(n, d_k)
    K = np.random.randn(n, d_k)
    V = np.random.randn(n, d_k)
    
    # Time multiple runs for stability
    run_times = []
    for _ in range(5):
        start = time.perf_counter()
        _ = scaled_dot_product_attention(Q, K, V)
        end = time.perf_counter()
        run_times.append(end - start)
    
    avg_time = np.median(run_times)
    times.append(avg_time)
    print(f"n={n:5d}  time={avg_time*1000:.3f} ms  attention_matrix_size={n*n:>10,}")

# Verify O(n^2) by checking the ratio
print("\n--- Ratio test (should be ~4x for each doubling of n) ---")
for i in range(1, len(times)):
    ratio = times[i] / times[i-1]
    n_ratio = seq_lengths[i] / seq_lengths[i-1]
    print(f"n: {seq_lengths[i-1]:4d} -> {seq_lengths[i]:4d}  "
          f"time ratio: {ratio:.2f}x  "
          f"(expected {n_ratio**2:.1f}x for O(n^2))")

In [ ]:
# Plot: actual runtime vs theoretical O(n^2) curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.plot(seq_lengths, [t * 1000 for t in times], 'bo-', label='Measured', linewidth=2)
# Fit O(n^2) curve
c = times[-1] / (seq_lengths[-1] ** 2)
theoretical = [c * n**2 * 1000 for n in seq_lengths]
ax1.plot(seq_lengths, theoretical, 'r--', label='O(n²) fit', linewidth=2)
ax1.set_xlabel('Sequence Length (n)')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Attention Runtime: Linear Scale')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log-log scale (O(n^2) should be a straight line with slope 2)
ax2.loglog(seq_lengths, times, 'bo-', label='Measured', linewidth=2)
ax2.loglog(seq_lengths, [c * n**2 for n in seq_lengths], 'r--', label='Slope=2 (O(n²))', linewidth=2)
ax2.set_xlabel('Sequence Length (n)')
ax2.set_ylabel('Time (seconds)')
ax2.set_title('Log-Log Scale (slope ≈ 2 confirms O(n²))')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInterview sentence: 'Attention scales as O(n² * d_k). "
      f"At n=2048 with d_k=64, the attention matrix alone has "
      f"{2048*2048:,} entries — {2048*2048*64:,} FLOPs per layer.'")

---

## Experiment 2: Failure Mode — Remove √d_k Scaling

**Claim to test:** Without √d_k scaling, dot products grow with d_k, causing softmax to saturate (push all weight to one token).

**Why it matters:** This is one of the most common Staff-level follow-ups: "What happens if you remove the scaling factor?"

In [ ]:
def attention_no_scaling(Q, K, V):
    """Attention WITHOUT the sqrt(d_k) scaling."""
    scores = Q @ K.T  # no division by sqrt(d_k)
    weights = softmax(scores)
    output = weights @ V
    return output, weights


# Compare scaled vs unscaled at increasing d_k
n = 10  # sequence length
d_k_values = [4, 16, 64, 256, 512, 1024]

print(f"{'d_k':>6}  {'Max weight (scaled)':>20}  {'Max weight (unscaled)':>22}  {'Entropy (scaled)':>18}  {'Entropy (unscaled)':>20}")
print("-" * 95)

max_weights_scaled = []
max_weights_unscaled = []
entropies_scaled = []
entropies_unscaled = []

for d_k in d_k_values:
    Q = np.random.randn(n, d_k)
    K = np.random.randn(n, d_k)
    V = np.random.randn(n, d_k)
    
    _, weights_scaled = scaled_dot_product_attention(Q, K, V)
    _, weights_unscaled = attention_no_scaling(Q, K, V)
    
    max_s = np.max(weights_scaled)
    max_u = np.max(weights_unscaled)
    
    # Entropy: higher = more spread out, lower = more concentrated
    entropy_s = -np.mean(np.sum(weights_scaled * np.log(weights_scaled + 1e-10), axis=-1))
    entropy_u = -np.mean(np.sum(weights_unscaled * np.log(weights_unscaled + 1e-10), axis=-1))
    
    max_weights_scaled.append(max_s)
    max_weights_unscaled.append(max_u)
    entropies_scaled.append(entropy_s)
    entropies_unscaled.append(entropy_u)
    
    print(f"{d_k:>6}  {max_s:>20.6f}  {max_u:>22.6f}  {entropy_s:>18.4f}  {entropy_u:>20.4f}")

In [ ]:
# Visualize the failure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(d_k_values, max_weights_scaled, 'go-', label='With √d_k scaling', linewidth=2)
ax1.plot(d_k_values, max_weights_unscaled, 'ro-', label='Without scaling', linewidth=2)
ax1.axhline(y=1/n, color='gray', linestyle=':', label=f'Uniform (1/{n})')
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='All weight on 1 token')
ax1.set_xlabel('d_k (key dimension)')
ax1.set_ylabel('Max attention weight')
ax1.set_title('Attention Saturation Without Scaling')
ax1.set_xscale('log', base=2)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(d_k_values, entropies_scaled, 'go-', label='With √d_k scaling', linewidth=2)
ax2.plot(d_k_values, entropies_unscaled, 'ro-', label='Without scaling', linewidth=2)
ax2.axhline(y=np.log(n), color='gray', linestyle=':', label=f'Max entropy (uniform)')
ax2.set_xlabel('d_k (key dimension)')
ax2.set_ylabel('Average entropy of attention weights')
ax2.set_title('Entropy Collapse Without Scaling')
ax2.set_xscale('log', base=2)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInterview sentence: 'Without √d_k scaling, at d_k=1024 the max attention weight "
      f"reaches {max_weights_unscaled[-1]:.4f} (nearly all weight on one token). "
      f"With scaling, it stays at {max_weights_scaled[-1]:.4f}. "
      f"The softmax saturates because QK^T dot products grow proportionally to d_k.'")

---

## Experiment 3: Ablation — Causal Mask vs No Mask

**Claim to test:** Causal masking fundamentally changes what attention can learn — each position can only look backward, creating an information asymmetry.

**Why it matters:** Understanding the difference between encoder (bidirectional) and decoder (causal) attention is critical for architecture choice questions.

In [ ]:
# Create a sentence and run both masked and unmasked attention
n = 8  # sequence length
d_k = 32

# Simulate embeddings
np.random.seed(42)
X = np.random.randn(n, d_k)

# Random projection matrices
W_Q = np.random.randn(d_k, d_k) * 0.1
W_K = np.random.randn(d_k, d_k) * 0.1
W_V = np.random.randn(d_k, d_k) * 0.1

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Bidirectional (encoder) attention
output_bidir, weights_bidir = scaled_dot_product_attention(Q, K, V)

# Causal (decoder) attention
causal_mask = np.triu(np.full((n, n), -np.inf), k=1)
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

# Compare outputs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im1 = axes[0].imshow(weights_bidir, cmap='Blues', vmin=0, vmax=0.4)
axes[0].set_title('Bidirectional (Encoder)\nEvery position sees everything')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(weights_causal, cmap='Reds', vmin=0, vmax=0.4)
axes[1].set_title('Causal (Decoder)\nEach position sees only past')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im2, ax=axes[1])

# Difference in output representations
diff = np.abs(output_bidir - output_causal)
im3 = axes[2].imshow(diff, cmap='YlOrRd', aspect='auto')
axes[2].set_title('|Output difference|\nHow much info is lost')
axes[2].set_xlabel('Dimension')
axes[2].set_ylabel('Position')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

# Quantify: how much does each position differ?
position_diff = np.linalg.norm(output_bidir - output_causal, axis=1)
print("Output difference by position (L2 norm):")
for i, d in enumerate(position_diff):
    bar = '█' * int(d * 10)
    print(f"  Position {i}: {d:.4f}  {bar}")

print(f"\nPosition 0 difference: {position_diff[0]:.4f} (sees same info in both — only itself)")
print(f"Position {n-1} difference: {position_diff[-1]:.4f} (biggest gap — bidirectional sees future)")
print(f"\nInterview sentence: 'Early positions have similar representations in both architectures. "
      f"Later positions diverge significantly because bidirectional attention sees future context.'")

---

## Experiment 4: Library Comparison — From-Scratch vs PyTorch

**Claim to test:** Our NumPy implementation produces the same result as PyTorch's scaled_dot_product_attention.

**Why it matters:** Validates that the from-scratch implementation is correct.

In [ ]:
try:
    import torch
    import torch.nn.functional as F
    
    n, d_k = 6, 64
    np.random.seed(42)
    Q_np = np.random.randn(n, d_k).astype(np.float32)
    K_np = np.random.randn(n, d_k).astype(np.float32)
    V_np = np.random.randn(n, d_k).astype(np.float32)
    
    # Our implementation
    output_np, weights_np = scaled_dot_product_attention(Q_np, K_np, V_np)
    
    # PyTorch implementation
    Q_pt = torch.from_numpy(Q_np).unsqueeze(0)  # add batch dim
    K_pt = torch.from_numpy(K_np).unsqueeze(0)
    V_pt = torch.from_numpy(V_np).unsqueeze(0)
    
    output_pt = F.scaled_dot_product_attention(Q_pt, K_pt, V_pt)
    output_pt_np = output_pt.squeeze(0).numpy()
    
    # Compare
    max_diff = np.max(np.abs(output_np - output_pt_np))
    mean_diff = np.mean(np.abs(output_np - output_pt_np))
    
    print(f"Max absolute difference:  {max_diff:.2e}")
    print(f"Mean absolute difference: {mean_diff:.2e}")
    print(f"Match: {'YES' if max_diff < 1e-5 else 'NO'} (threshold: 1e-5)")
    print(f"\nOur NumPy implementation matches PyTorch's scaled_dot_product_attention.")
    
except ImportError:
    print("PyTorch not installed. Skipping library comparison.")
    print("Install with: pip install torch")
    print("\nThe NumPy implementation follows the exact formula from the paper:")
    print("  Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V")
    print("This is mathematically identical to any correct implementation.")

---

## Experiment 5: Variance Analysis — Why √d_k Works

**Claim to test:** When Q and K entries are drawn from N(0,1), the variance of QK^T entries is d_k. Dividing by √d_k normalizes variance to 1.

**Why it matters:** This derivation is the most common follow-up in attention interviews. Having empirical evidence alongside the math makes your answer airtight.

In [ ]:
# Measure the variance of dot products at different d_k values
d_k_values = [4, 8, 16, 32, 64, 128, 256, 512]
n_samples = 10000

variances_raw = []
variances_scaled = []

print(f"{'d_k':>6}  {'Var(QK^T)':>12}  {'Expected (=d_k)':>16}  {'Var(QK^T/√d_k)':>16}  {'Expected (=1)':>14}")
print("-" * 70)

for d_k in d_k_values:
    # Generate many random Q, K vectors and compute their dot products
    Q = np.random.randn(n_samples, d_k)
    K = np.random.randn(n_samples, d_k)
    
    # Raw dot products
    dots = np.sum(Q * K, axis=1)  # each entry is q_i . k_i
    var_raw = np.var(dots)
    
    # Scaled dot products
    dots_scaled = dots / np.sqrt(d_k)
    var_scaled = np.var(dots_scaled)
    
    variances_raw.append(var_raw)
    variances_scaled.append(var_scaled)
    
    print(f"{d_k:>6}  {var_raw:>12.2f}  {d_k:>16}  {var_scaled:>16.4f}  {1.0:>14}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(d_k_values, variances_raw, 'bo-', label='Measured Var(QK^T)', linewidth=2)
ax1.plot(d_k_values, d_k_values, 'r--', label='Theoretical: Var = d_k', linewidth=2)
ax1.set_xlabel('d_k')
ax1.set_ylabel('Variance of dot products')
ax1.set_title('Without Scaling: Variance Grows with d_k')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(d_k_values, variances_scaled, 'go-', label='Measured Var(QK^T/√d_k)', linewidth=2)
ax2.axhline(y=1.0, color='r', linestyle='--', label='Theoretical: Var = 1', linewidth=2)
ax2.set_xlabel('d_k')
ax2.set_ylabel('Variance of scaled dot products')
ax2.set_title('With √d_k Scaling: Variance Stays at 1')
ax2.set_ylim(0, 2)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInterview sentence: 'When Q and K are N(0,1), the dot product QK^T has variance d_k "
      f"because it sums d_k products of independent N(0,1) variables. Dividing by √d_k "
      f"normalizes variance to 1, keeping softmax in its useful gradient region.'")

---

## Summary

**Claims now backed by evidence:**

1. Attention runtime scales as O(n²) — confirmed with log-log slope ≈ 2
2. Without √d_k scaling, softmax saturates — max weight approaches 1.0 at large d_k
3. Causal masking creates an information asymmetry — later positions are most affected
4. Our implementation matches PyTorch's output (or follows the exact paper formula)
5. Variance of QK^T is exactly d_k — dividing by √d_k normalizes to 1

For the full mathematical treatment and interview Q&A, see [attention-mechanisms-interview.md](./attention-mechanisms-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)